# Tutorial: Train and evaluate a reducer selector

This tutorial covers the supervised learning workflow: load recorded data, train a selector, evaluate it, and save a checkpoint. It uses recorded teacher data, so simulation and teacher collection are not required.

Choose **Run → Run All Cells** to execute the complete workflow.

The network scores four sound native reducer actions: `girard`, `scott`, `pca`, `combastel`. Lower scores rank first. The native execution layer handles bounds and fallbacks. Training uses the existing pairwise objective, rather than predicting calibrated costs.

The starter contains budget-40 decision states from training seeds 0–3 and validation seeds 20–21. Validation is for development, not a fresh generalization claim.

In [ ]:
from pzr import learning
import pandas as pd
import torch

torch.set_num_threads(1)
task = learning.load_task("robot_arm", bundle="starter", budget=40)

In [ ]:
pd.DataFrame({"split": ["train", "validation"],
              "states": [task.training.num_samples, task.validation.num_samples]})

## Train and inspect validation results
The tutorial uses twenty epochs to limit runtime. The returned policy uses the best validation checkpoint.

In [ ]:
run = learning.train_selector(task, config=learning.TrainingConfig(epochs=20, seed=42))
pd.Series(learning.evaluate_selector(run.policy, task.validation))

## Compare with the frozen reference
The frozen Clean148 model used 148 training trajectories. This is a reference, not a controlled comparison with the four-trajectory starter model. Notebook 01 compares two networks trained on identical data.

In [ ]:
pd.DataFrame({
    "starter MLP": learning.evaluate_selector(run.policy, task.validation),
    "frozen Clean148 (more training data)": learning.evaluate_selector(task.frozen_baseline(), task.validation),
})

## Run in the native monitor
This short prefix checks integration. Prefix losses are explicitly labelled; remove `max_events` for full 500-event validation trajectories. Exact references may take longer than neural training.

In [ ]:
learning.evaluate_monitor(run.policy, task.validation_traces, max_events=20,
                         reference_cache="workspace/references")

## Save and reload
Each execution uses a new directory, so rerunning the notebook preserves earlier experiments.

In [ ]:
from pathlib import Path
from uuid import uuid4
output = Path("workspace") / ("first-selector-" + uuid4().hex[:8])
run.save(output)
reloaded = learning.load_selector(output)
print(output)
assert learning.evaluate_selector(reloaded, task.validation) == learning.evaluate_selector(run.policy, task.validation)